# Smoke test + how much do we still need to label?

This notebook runs the full chain on the current Roboflow export and then estimates how much labeling is still needed.

1. Environment check
2. Time-based train/val split
3. Label coverage over the 660-frame clip
4. Training (fine-tune YOLO26s)
5. Evaluation per class
6. **How much still needs labeling?**
   - 6.1 run the model over every video frame (+ save the annotated video)
   - 6.2 count-based estimate: extra frames needed to reach a target number of instances per class
   - 6.3 which frames to label next (`label_candidates.csv`)
   - 6.4 optional learning curve: does more data still help?

Run it with the `venv` kernel. Change the settings in the *Settings* cell. The notebook only writes inside `kobe/smoke_test/`.

In [ ]:
import csv
import math
import re
import runpy
import sys
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml
from ultralytics import YOLO

# The notebook must run from kobe/smoke_test/, where common.py lives
assert (Path.cwd() / "common.py").exists(), "Run this notebook from kobe/smoke_test/"
sys.path.insert(0, str(Path.cwd()))
from common import (BEST_WEIGHTS, DATA_YAML, DATASET_DIR, REPO_ROOT, ROBOFLOW_YAML, RUNS_DIR,
                    RUN_NAME, SMOKE_DIR, VIDEO_PATH, disable_label_cache, pick_device)

## Settings

In [ ]:
RUN_TRAINING = True         # False = reuse the existing runs/smoke/weights/best.pt
MODEL = "yolo26m.pt"        # pretrained medium model (YOLO26 needs ultralytics >= 8.4)
EPOCHS, IMGSZ, BATCH = 100, 1600, 2   # RTX 4070 training settings
PATIENCE, MOSAIC, CLOSE_MOSAIC = 20, 0.2, 20

SAVE_VIDEO = True           # write the annotated video during the pass in 6.1
RUN_LEARNING_CURVE = False  # 6.4 trains extra models -> slow on CPU
LC_FRACTIONS = [0.25, 0.5, 1.0]

# Target number of labeled instances per class (our own assumption, see 6.2)
TARGET = {"train": 300, "val": 50}
MIN_GAP = 3                 # min. distance (frames) between frames we pick to label

# Split rule (frame index in the 660-frame clip)
TRAIN_MAX, VAL_MIN, N_FRAMES = 494, 525, 660

DEVICE = pick_device()      # GPU 0 if available, else "cpu" (6 threads)
disable_label_cache()       # never write labels.cache into data/
print("device:", DEVICE)

## 1. Environment

In [ ]:
runpy.run_path("check_env.py")

## 2. Split
Runs `make_split.py`: train = idx 0–494, gap = 495–524 (never used), val = idx 525–659.

In [ ]:
runpy.run_path("make_split.py")

In [ ]:
names = yaml.safe_load(ROBOFLOW_YAML.read_text())["names"]  # class names from the Roboflow export


def frame_idx(path):
    """Original frame index from a name like frame_0000_idx_00000_jpg.rf.<hash>.txt"""
    return int(re.search(r"idx_(\d+)", Path(path).name).group(1))


def split_of(idx):
    return "train" if idx <= TRAIN_MAX else "val" if idx >= VAL_MIN else "gap"


# labeled[idx] = Counter {class id: number of boxes} for every labeled frame
labeled = {}
for label_file in sorted((DATASET_DIR / "train" / "labels").glob("*.txt")):
    ids = [int(line.split()[0]) for line in label_file.read_text().splitlines() if line.strip()]
    labeled[frame_idx(label_file)] = Counter(ids)
print(len(labeled), "labeled frames")

## 3. Label coverage over the clip

In [ ]:
fig, ax = plt.subplots(figsize=(12, 1.8))
ax.axvspan(0, TRAIN_MAX + 1, color="tab:blue", alpha=0.1, label="train zone")
ax.axvspan(TRAIN_MAX + 1, VAL_MIN, color="grey", alpha=0.3, label="gap")
ax.axvspan(VAL_MIN, N_FRAMES, color="tab:orange", alpha=0.1, label="val zone")
ax.vlines(list(labeled), 0, 1, color="black", lw=1)  # one line per labeled frame
ax.set(xlim=(0, N_FRAMES), yticks=[], xlabel="frame index",
       title=f"Labeled frames: {len(labeled)} of {N_FRAMES}")
ax.legend(loc="upper right", ncol=3, fontsize=8)
plt.show()

for split, (lo, hi) in {"train": (0, TRAIN_MAX), "val": (VAL_MIN, N_FRAMES - 1)}.items():
    n_zone = hi - lo + 1
    n_lab = sum(split_of(i) == split for i in labeled)
    print(f"{split:<6} zone {lo}-{hi}: {n_lab}/{n_zone} frames labeled ({100 * n_lab / n_zone:.0f} %)")

## 4. Training
Fine-tunes the pretrained model on the train split. On CPU this takes a while (see the printed time).

In [ ]:
if RUN_TRAINING:
    t0 = time.time()
    YOLO(str(SMOKE_DIR / MODEL)).train(
        data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=DEVICE,
        patience=PATIENCE, mosaic=MOSAIC, close_mosaic=CLOSE_MOSAIC,
        project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True, seed=0, plots=True, workers=0,
    )
    print(f"training took {(time.time() - t0) / 60:.1f} min")

model = YOLO(str(BEST_WEIGHTS))


def show(path, size=(12, 6)):
    """Display a saved image (plot) inside the notebook."""
    plt.figure(figsize=size)
    plt.imshow(plt.imread(str(path)))
    plt.axis("off")
    plt.title(Path(path).name)
    plt.show()


show(RUNS_DIR / RUN_NAME / "results.png")

## 5. Evaluation per class (val split)

In [ ]:
def evaluate(weights, run_name):
    """Validate on the val split. Returns {class name: (P, R, mAP50, mAP50-95)} plus "ALL"."""
    m = YOLO(str(weights))
    box = m.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                project=str(RUNS_DIR), name=run_name, exist_ok=True, plots=True, verbose=False).box
    # class_result(i) takes the position in ap_class_index, not the class id
    out = {m.names[int(cid)]: box.class_result(i) for i, cid in enumerate(box.ap_class_index)}
    out["ALL"] = (box.mp, box.mr, box.map50, box.map)
    return out


scores = evaluate(BEST_WEIGHTS, f"{RUN_NAME}_val")
print(f"{'class':<22}{'P':>7}{'R':>7}{'mAP50':>8}{'mAP50-95':>10}")
for name, (p, r, ap50, ap) in scores.items():
    print(f"{name:<22}{p:>7.3f}{r:>7.3f}{ap50:>8.3f}{ap:>10.3f}")

show(RUNS_DIR / f"{RUN_NAME}_val" / "confusion_matrix_normalized.png", (8, 6))
show(RUNS_DIR / f"{RUN_NAME}_val" / "val_batch0_pred.jpg")

## 6. How much still needs labeling?

### 6.1 Model pass over all 660 video frames
For every frame we store: split zone, already labeled or not, whether it is an exact **duplicate** of the previous frame
(the clip repeats frames, a duplicate adds no information), and the highest **ball confidence**.
A low ball confidence means the model is unsure → such frames are the most useful to label.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS)
video_out = RUNS_DIR / f"{RUN_NAME}_predict" / "match_pred.mp4"
writer = None
rows = []      # one dict per video frame
prev = None    # previous frame (small grey copy) for the duplicate check
t0 = time.time()

while True:
    ok, frame = cap.read()
    if not ok:
        break
    idx = len(rows)  # frame index = position in the video

    # Duplicate: almost no pixel difference with the previous frame
    small = cv2.cvtColor(cv2.resize(frame, (480, 270)), cv2.COLOR_BGR2GRAY).astype(np.float32)
    duplicate = prev is not None and np.abs(small - prev).mean() < 0.5
    prev = small

    res = model.predict(frame, imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)[0]
    classes = res.boxes.cls.int().tolist()
    confs = res.boxes.conf.tolist()
    ball_confs = [c for k, c in zip(classes, confs) if model.names[k] == "ball"]
    rows.append({"idx": idx, "split": split_of(idx), "labeled": idx in labeled,
                 "duplicate": duplicate, "ball_conf": max(ball_confs, default=0.0)})

    if SAVE_VIDEO:
        if writer is None:  # open the writer once we know the frame size
            video_out.parent.mkdir(parents=True, exist_ok=True)
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(str(video_out), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        writer.write(res.plot())  # frame with boxes drawn on it

cap.release()
if writer is not None:
    writer.release()
    print("annotated video:", video_out)
n_dup = sum(r["duplicate"] for r in rows)
print(f"{len(rows)} frames in {(time.time() - t0) / 60:.1f} min, {n_dup} exact duplicates")

In [ ]:
x = [r["idx"] for r in rows]
fig, ax = plt.subplots(figsize=(12, 3))
ax.axvspan(TRAIN_MAX + 1, VAL_MIN, color="grey", alpha=0.3, label="gap")
ax.plot(x, [r["ball_conf"] for r in rows], lw=0.8, label="max ball confidence (0 = no ball found)")
ax.plot(list(labeled), [1.02] * len(labeled), "k|", label="labeled frame")
ax.set(xlim=(0, N_FRAMES), ylim=(0, 1.05), xlabel="frame index")
ax.legend(loc="lower right", fontsize=8)
plt.show()

for split in ["train", "val"]:
    zone = [r for r in rows if r["split"] == split and not r["labeled"]]
    found = sum(r["ball_conf"] > 0 for r in zone)
    print(f"{split}: ball found in {found}/{len(zone)} unlabeled frames")

### 6.2 Count-based estimate

Assumption: we want at least `TARGET[split]` labeled instances per class.
Why a target for **val**: with N instances, one hit or miss changes the recall by 100/N %.
Now there are 4 balls in val → 25 % per ball. With 50 instances it is 2 %.

Per class: *instances per labeled frame* (from the current labels) → extra frames needed.
One labeled frame labels all classes at once, so per split the **rarest class decides**.
Duplicates and the gap are not counted as available frames.

In [ ]:
needed = {}  # extra frames needed per split
for split in ["train", "val"]:
    lab_frames = [i for i in labeled if split_of(i) == split]
    free = [r for r in rows if r["split"] == split and not r["labeled"] and not r["duplicate"]]
    print(f"\n=== {split}: {len(lab_frames)} labeled, {len(free)} unlabeled non-duplicate frames left,"
          f" target {TARGET[split]} instances/class ===")
    print(f"{'class':<22}{'have':>6}{'per frame':>11}{'extra frames':>14}{'max in clip':>13}")
    worst = 0
    for cid, name in enumerate(names):
        have = sum(labeled[i][cid] for i in lab_frames)
        rate = have / len(lab_frames) if lab_frames else 0.0    # instances per labeled frame
        if rate == 0:
            print(f"{name:<22}{have:>6}   no labels yet -> cannot estimate")
            continue
        extra = math.ceil(max(0, TARGET[split] - have) / rate)
        max_clip = have + rate * len(free)                      # if we label everything left
        worst = max(worst, extra)
        print(f"{name:<22}{have:>6}{rate:>11.2f}{extra:>14}{max_clip:>13.0f}")
    needed[split] = worst
    if worst > len(free):
        print(f"-> NOT reachable in this clip: {worst} frames needed, only {len(free)} left."
              " Lower the target or add more video.")
    else:
        print(f"-> label about {worst} more frames ({100 * worst / len(free):.0f} % of what is left)")

### 6.3 Which frames to label next?

From the unlabeled, non-duplicate frames of each split we pick the frames where the model is **least sure about the ball**
(lowest ball confidence first), keeping at least `MIN_GAP` frames between picks because neighbouring frames are near-identical.
The result goes to `label_candidates.csv` (priority 1 = label first). The file names refer to `data/frames/`.

Tip: label in rounds (e.g. 30 frames), retrain, and run this notebook again: the ranking improves as the model improves.

Caveat: if the model finds the ball in (almost) no frame, all confidences are 0 and the ranking is simply chronological
(every `MIN_GAP`-th frame). That is still a reasonable spread, but it is not "uncertainty-based".

In [ ]:
def pick_candidates(split):
    """Unlabeled, non-duplicate frames of a split, lowest ball confidence first, spread in time."""
    pool = [r for r in rows if r["split"] == split and not r["labeled"] and not r["duplicate"]]
    pool.sort(key=lambda r: r["ball_conf"])
    taken, chosen = set(labeled), []
    for r in pool:
        if all(abs(r["idx"] - t) >= MIN_GAP for t in taken):
            chosen.append(r)
            taken.add(r["idx"])
    return chosen


csv_path = SMOKE_DIR / "label_candidates.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer_csv = csv.writer(f)
    writer_csv.writerow(["split", "priority", "frame_idx", "ball_conf", "file"])
    for split in ["train", "val"]:
        chosen = pick_candidates(split)
        for prio, r in enumerate(chosen, start=1):
            name = f"frame_{r['idx']:04d}_idx_{r['idx']:05d}.jpg"
            exists = (REPO_ROOT / "data" / "frames" / name).exists()
            writer_csv.writerow([split, prio, r["idx"], round(r["ball_conf"], 3),
                                 f"data/frames/{name}" if exists else ""])
        print(f"{split}: {len(chosen)} candidates (MIN_GAP={MIN_GAP}), estimate says {needed[split]} needed")
        print("  first 20 by priority:", [r["idx"] for r in chosen[:20]])
print("written:", csv_path)

### 6.4 Learning curve (optional, `RUN_LEARNING_CURVE = True`)

Train the same model on 25 %, 50 % and 100 % of the train frames (spread evenly in time) and evaluate each on the same val split.
If mAP still rises clearly from 50 % → 100 %, more labels will help. If the curve is flat, more of the *same kind* of frames
helps little (then label different situations, or a larger image size for the ball).
With only 4 val images the curve is noisy — read it as a trend, not as exact numbers.

In [ ]:
if RUN_LEARNING_CURVE:
    train_imgs = (SMOKE_DIR / "train.txt").read_text().splitlines()  # paths contain spaces
    curve = {}
    for frac in LC_FRACTIONS:
        n = max(1, round(len(train_imgs) * frac))
        if n == len(train_imgs):
            curve[n] = scores  # 100 % = the model from section 4
            continue
        # n images evenly spread over time (not just the first n)
        pick = np.linspace(0, len(train_imgs) - 1, n).round().astype(int)
        (SMOKE_DIR / f"train_lc_{n}.txt").write_text("\n".join(train_imgs[i] for i in pick) + "\n")
        cfg = yaml.safe_load(DATA_YAML.read_text())
        cfg["train"] = f"train_lc_{n}.txt"
        lc_yaml = SMOKE_DIR / f"data_lc_{n}.yaml"
        lc_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))

        YOLO(str(SMOKE_DIR / MODEL)).train(
            data=str(lc_yaml), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=DEVICE,
            patience=PATIENCE, mosaic=MOSAIC, close_mosaic=CLOSE_MOSAIC,
            project=str(RUNS_DIR), name=f"lc_{n}", exist_ok=True, seed=0, plots=False, workers=0,
        )
        curve[n] = evaluate(RUNS_DIR / f"lc_{n}" / "weights" / "best.pt", f"lc_{n}_val")

    sizes = sorted(curve)
    for name in names + ["ALL"]:
        plt.plot(sizes, [curve[n].get(name, (0, 0, 0, 0))[2] for n in sizes], "o-", label=name)
    plt.xlabel("number of train images")
    plt.ylabel("mAP@50 on val")
    plt.legend(fontsize=8)
    plt.show()

## 7. Conclusion (run of 2026-10-19: YOLO26s, imgsz 640, 10 epochs, CPU, 22.7 min training)

**The chain works**: Roboflow export → split → training → evaluation → video prediction → labeling estimate.

| class | P | R | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|
| ball | 0.000 | 0.000 | 0.000 | 0.000 |
| player_A | 0.767 | 0.875 | 0.863 | 0.399 |
| player_B | 0.748 | 0.575 | 0.667 | 0.215 |
| player_B_goal_keeper | 1.000 | 0.000 | 0.781 | 0.263 |
| referee | 1.000 | 0.000 | 0.016 | 0.002 |
| **all** | 0.703 | 0.290 | 0.466 | 0.176 |

- **Ball: never detected** (not in val, and in none of the 579 unlabeled frames). The ball is ≈ 5 px at imgsz 640,
  and 42 training instances is far too little. More labels alone will probably not fix this: a larger `imgsz` (1280)
  or tiling/SAHI is needed as well.
- **Players** are found reasonably well, but the confusion matrix shows many player_A ↔ player_B mix-ups and false
  positives (e.g. on the advertising boards).
- Goalkeeper/referee numbers are based on **4 val instances each** → not meaningful.
- **Still to label (targets 300 train / 50 val per class):** ≈ 46 val frames and ≈ 289 train frames.
  The rarest classes (ball, goalkeeper, referee in val) decide this. For train that is 80 % of the remaining
  non-duplicate frames, so the targets are close to what this clip can give at all.
- Because the model never finds the ball, `label_candidates.csv` is now a **time-spread list** (every 3rd frame),
  not an uncertainty ranking. Priority for the team: first the val frames (a reliable val set), then train in rounds.